# Step 2 - Train/test split and word2vec utterance vectors (paper Sec. 3.2-3.3, Table 2)

* Reads the cleaned transcripts (`./clean`, `./clean_control`) and `id_label.txt`.
* Keeps the four classes with enough subjects: Anomic (0), Broca (1), Conduction (2), Control (3).
* **Patient-wise split**: the first 30 subjects of each class go to the test set, the rest to training.
* Keeps utterances with more than 5 tokens; the utterance label is the subject's label.
* Each utterance becomes a 28 x 300 matrix of GoogleNews word2vec vectors
  (OOV words are skipped), zero-padded to 28 rows.
* Saves `aphasia_np_{labels,labels_num,text,subj_id,vectors}_patientwise_{train,test}.np.npy`.

Needs `GoogleNews-vectors-negative300.bin` (https://code.google.com/archive/p/word2vec/).
Cells were run out of order: the word2vec model loaded in the later cell is used by the vectorization cell.
Cells after the patient-wise part are an earlier utterance-wise random split (`full5`) that is not used in the paper.

*Original file name: `DataVectors_preprocess.ipynb`. Code cells are unchanged from 2019/2020; outputs were cleared and these markdown notes were added in 2026.*

## Load cleaned transcripts and labels

In [ ]:
# imports
import os
# imports
import gensim
import nltk
from random import shuffle
import numpy as np

In [ ]:
patients = []
controls = []


for root, dirs, files in os.walk("./clean"):
    for filename in files:
        if filename[-5:] == "clean":
            patients.append(root+'/'+filename)

        
for root, dirs, files in os.walk("./clean_control"):
    for filename in files:
        if filename[-5:] == "clean":
            controls.append(root+'/'+filename)

        
label_file = "id_label.txt"

print("# of patients: ", len(patients))
print("# of controls: ", len(controls))
print("# of subjects: ", len(patients)+len(controls))

In [ ]:
# per patient
# patient_dict: {(patient_id, label): text}
# label_dict: {label: (patient_id, text)}
# id_label: {patient_id: label}

patient_dict = {}
label_dict = {}
id_label = {}


with open(label_file, encoding="utf-8") as lf:
    patient_label = lf.read().split('\n')

for line in patient_label:
    _id, label = line.split("\t")
    id_label[_id] = label


for p in patients:
    with open(p) as file:
        p_text = file.read()
    _id = p[:-6].split('/')[2]
    label = id_label[_id]
    patient_dict[(_id, label)] = p_text
    if label not in label_dict:
        label_dict[label] = [(_id, p_text.split('\n'))]
    else:
        label_dict[label].append((_id, p_text.split("\n")))
    

for c in controls:
    with open(c) as file:
        c_text = file.read()
    _id = c[:-6].split('/')[2]
    label = id_label[_id]
    patient_dict[(_id, label)] = c_text
    if label not in label_dict:
        label_dict[label] = [(_id, c_text.split('\n'))]
    else:
        label_dict[label].append((_id, c_text.split('\n')))

In [ ]:
# Devide data (phrases) patientwise 
for i in label_dict:
    print(i, len(label_dict[i]))

In [ ]:
label_dict["TransSensory"]

## Patient-wise train/test split (30 test subjects per class)

In [ ]:
# big enough labels dict
by_patieln_ld = {
    "Anomic": 0,
    #"Wernicke": 1,
    "Broca": 1,
    "Conduction": 2,
    "Control": 3
}

for i in label_dict:
    if i in by_patieln_ld:
        print(i, len(label_dict[i]))

In [ ]:
test_set = []
train_set = []
for l in label_dict:
    if l in by_patieln_ld:
        test = label_dict[l][:30]
        train = label_dict[l][30:]
        test_set.append((test, l))
        train_set.append((train, l))

In [ ]:
tr_label_arr = []
tr_label_num_arr = []
tr_text_arr = []
tr_subj_id = []
tr_matrix_arr = []
for l in train_set:
    label = l[1]
    label_num = by_patieln_ld[label]
    data = l[0]
    for subj in data:
        _id = subj[0]
        text = subj[1]
        for phrase in text:
            tokens = phrase.split()
            if len(tokens) > 5:
                tr_label_arr.append(label)
                tr_label_num_arr.append(label_num)
                tr_text_arr.append(phrase)
                tr_subj_id.append(_id)

In [ ]:
np.save("aphasia_np_labels_patientwise_train.np", np.array(tr_label_arr))
np.save("aphasia_np_labels_num_patientwise_train.np", np.array(tr_label_num_arr))
np.save("aphasia_np_text_patientwise_train.np", np.array(tr_text_arr))
np.save("aphasia_np_subj_id_patientwise_train.np", np.array(tr_subj_id))

In [ ]:
te_label_arr = []
te_label_num_arr = []
te_text_arr = []
te_subj_id = []
te_matrix_arr = []
for l in test_set:
    label = l[1]
    label_num = by_patieln_ld[label]
    data = l[0]
    for subj in data:
        _id = subj[0]
        text = subj[1]
        for phrase in text:
            tokens = phrase.split()
            if len(tokens) > 5:
                te_label_arr.append(label)
                te_label_num_arr.append(label_num)
                te_text_arr.append(phrase)
                te_subj_id.append(_id)

In [ ]:
np.save("aphasia_np_labels_patientwise_test.np", np.array(te_label_arr))
np.save("aphasia_np_labels_num_patientwise_test.np", np.array(te_label_num_arr))
np.save("aphasia_np_text_patientwise_test.np", np.array(te_text_arr))
np.save("aphasia_np_subj_id_patientwise_test.np", np.array(te_subj_id))

In [ ]:
len_arr = []
for i in test_data:
    len_arr.append(len(i))
for i in train_data:
    len_arr.append(len(i))
max(len_arr)

## word2vec matrices, zero padding to 28 x 300, saving

In [ ]:
test_data = []
counter = 0
for phrase in te_text_arr:
    counter += 1
    print(counter, '/', len(te_text_arr))
    matrix = []
    tokens = nltk.word_tokenize(phrase)
    for t in tokens:
        try:
            v = model[t]
            matrix.append(v)
        except:
            pass
    test_data.append(matrix)

counter = 0

train_data = []
counter = 0
for phrase in tr_text_arr:
    counter += 1
    print(counter, '/', len(tr_text_arr))
    matrix = []
    tokens = nltk.word_tokenize(phrase)
    for t in tokens:
        try:
            v = model[t]
            matrix.append(v)
        except:
            pass
    train_data.append(matrix)

In [ ]:
for n in range(len(test_data)):
    mx = test_data[n]
    empt = [[0]*300]*(28-len(mx))
    mx += empt
    test_data[n] = mx
    
for n in range(len(train_data)):
    mx = train_data[n]
    empt = [[0]*300]*(28-len(mx))
    mx += empt
    train_data[n] = mx

In [ ]:
test_np_vectors = np.array([test_data[0]])
print(test_np_vectors.shape)
c = 0
for i in test_data:
    c += 1
    print(c, '/', len(test_data))
    #print(len(i), len(i[0]))
    test_np_vectors = np.append(test_np_vectors, [i], axis=0)

In [ ]:
train_np_vectors1 = np.array([test_data[0]])
print(train_np_vectors1.shape)
c = 0
for i in train_data[:10000]:
    c += 1
    print(c, '/', 10000)
    #print(len(i), len(i[0]))
    train_np_vectors1 = np.append(train_np_vectors1, [i], axis=0)

In [ ]:
train_np_vectors2 = np.array([test_data[0]])
print(train_np_vectors2.shape)
c = 0
for i in train_data[10000:20000]:
    c += 1
    print(c, '/', 10000)
    #print(len(i), len(i[0]))
    train_np_vectors2 = np.append(train_np_vectors2, [i], axis=0)

In [ ]:
train_np_vectors3 = np.array([test_data[0]])
print(train_np_vectors3.shape)
c = 0
for i in train_data[20000:30000]:
    c += 1
    print(c, '/', 10000)
    #print(len(i), len(i[0]))
    train_np_vectors3 = np.append(train_np_vectors3, [i], axis=0)

In [ ]:
train_np_vectors4 = np.array([test_data[0]])
print(train_np_vectors4.shape)
c = 0
for i in train_data[30000:40000]:
    c += 1
    print(c, '/', 10000)
    #print(len(i), len(i[0]))
    train_np_vectors4 = np.append(train_np_vectors4, [i], axis=0)

In [ ]:
train_np_vectors5 = np.array([test_data[0]])
print(train_np_vectors5.shape)
c = 0
for i in train_data[40000:]:
    c += 1
    print(c, '/', 10000)
    #print(len(i), len(i[0]))
    train_np_vectors5 = np.append(train_np_vectors5, [i], axis=0)

In [ ]:
train_np_vectors = np.array([test_data[0]])

In [ ]:
train_np_vectors = np.append(train_np_vectors, train_np_vectors5[1:], axis=0)

In [ ]:
train_np_vectors = train_np_vectors[1:]

In [ ]:
len(train_np_vectors)

In [ ]:
np.save("aphasia_np_vectors_patientwise_train.np", train_np_vectors)

In [ ]:
np.save("aphasia_np_vectors_patientwise_test.np", test_np_vectors)

## Earlier experiment: utterance-wise random split (`full5`), not used in the paper

In [ ]:
# data_per_line: [patient_id, label, line]
data_per_line = []

for idl in patient_dict:
    p_text = patient_dict[idl]
    for l in p_text.split('\n'):
        data_per_line.append(list(idl)+[l])
#len(data_per_line)

In [ ]:
lengths = []
for i in data_per_line:
    t = i[2]
    lengths.append(len(t))
print(max(lengths))

In [ ]:
labels = []
labels_count = {}
for i in data_per_line:
    labels.append(i[1])
    if i[1] not in labels_count:
        labels_count[i[1]] = 1
    else:
        labels_count[i[1]] += 1
labels_count

In [ ]:
{'Anomic': 23605,
 'Wernicke': 7081,
 'Broca': 14120,
 'Conduction': 14516,
 'TransMotor': 1153,
 'Control': 46044}

In [ ]:
1: 10000, 2: 5000, 3: 5000, 4: 15000

In [ ]:
labels_dict = {
    'Anomic': 0,
    'Wernicke': 1,
    'Broca': 2,
    'Conduction': 3,
    'TransMotor': 4,
    #'TransSensory': 5,
    #'Global': 6,
    'Control': 7
    }

In [ ]:
data = []
for line in data_per_line:
    _id, label, text = line
    if label in labels_dict:
        words = text.split()
        if len(words) > 5:
            data.append((text, labels_dict[label]))

In [ ]:
len(data)

In [ ]:
labels_count = {}
labels_texts = {}
for i in data:
    if i[1] not in labels_count:
        labels_count[i[1]] = 1
        labels_texts[i[1]] = [i[0]]
    else:
        labels_count[i[1]] += 1
        labels_texts[i[1]].append([i[0]])
labels_count

In [ ]:
test_data = []
test_labels = []
train_data = []
train_labels = []
for label in labels_texts:
    texts = labels_texts[label]
    shuffle(texts)
    test = texts[:200]
    train = texts[200:]
    for test_dp in test:
        test_data.append(test_dp[0])
        test_labels.append(label)
    for train_dp in train:
        train_data.append(train_dp[0])
        train_labels.append(label)

In [ ]:
len(test_data), len(test_labels), len(train_data), len(train_labels)

In [ ]:
test_labels[0]

In [ ]:
model = gensim.models.KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin', binary=True) 

In [ ]:
test_matrix = []
for n in range(len(test_data)):
    line = test_data[n]
    label = test_labels[n]
    matrix = []
    words = nltk.word_tokenize(line)
    for w in words:
        try:
            vec = model[w]
            matrix.append(vec)
        except:
            pass
    test_matrix.append(matrix)
    print(n, '/', len(test_data))

In [ ]:
train_matrix = []
for n in range(len(train_data)):
    line = train_data[n]
    label = train_labels[n]
    matrix = []
    words = nltk.word_tokenize(line)
    for w in words:
        try:
            vec = model[w]
            matrix.append(vec)
        except:
            pass
    train_matrix.append(matrix)
    print(n, '/', len(train_data))

In [ ]:
# finding max len
len_arr = []
for i in test_matrix:
    len_arr.append(len(i))
for i in train_matrix:
    len_arr.append(len(i))
max(len_arr)

In [ ]:
for n in range(len(test_matrix)):
    mx = test_matrix[n]
    empt = [[0]*300]*(28-len(mx))
    mx += empt
    test_matrix[n] = mx
    
for n in range(len(train_matrix)):
    mx = train_matrix[n]
    empt = [[0]*300]*(28-len(mx))
    mx += empt
    train_matrix[n] = mx

In [ ]:
test_np_vectors = np.array([test_matrix[0]])
print(test_np_vectors.shape)
c = 0
for i in test_matrix:
    c += 1
    print(c, '/', len(test_matrix))
    #print(len(i), len(i[0]))
    test_np_vectors = np.append(test_np_vectors, [i], axis=0)

In [ ]:
train_np_vectors = np.array([train_matrix[0]])
print(train_np_vectors.shape)
c = 0
for i in train_matrix:
    c += 1
    print(c, '/', len(train_matrix))
    #print(len(i), len(i[0]))
    train_np_vectors = np.append(train_np_vectors, [i], axis=0)

In [ ]:
train_np_vectors[:20000][-1]

In [ ]:
train_matrix1 = train_matrix[:10000]
train_matrix2 = train_matrix[10000:20000]
train_matrix3 = train_matrix[20000:30000]
train_matrix4 = train_matrix[30000:40000]
train_matrix5 = train_matrix[40000:50000]
train_matrix6 = train_matrix[50000:60000]
train_matrix7 = train_matrix[60000:]

In [ ]:
train_np_vectors1 = np.array([train_matrix[0]])
print(train_np_vectors.shape)
c = 0
for i in train_matrix1:
    c += 1
    print(c, '/', 10000)
    #print(len(i), len(i[0]))
    train_np_vectors1 = np.append(train_np_vectors1, [i], axis=0)

In [ ]:
train_np_vectors2 = np.array([train_matrix[0]])
print(train_np_vectors.shape)
c = 0
for i in train_matrix2:
    c += 1
    print(c, '/', 10000)
    #print(len(i), len(i[0]))
    train_np_vectors2 = np.append(train_np_vectors2, [i], axis=0)

In [ ]:
train_np_vectors3 = np.array([train_matrix[0]])
print(train_np_vectors.shape)
c = 0
for i in train_matrix3:
    c += 1
    print(c, '/', 10000)
    #print(len(i), len(i[0]))
    train_np_vectors3 = np.append(train_np_vectors3, [i], axis=0)

In [ ]:
train_np_vectors4 = np.array([train_matrix[0]])
print(train_np_vectors.shape)
c = 0
for i in train_matrix4:
    c += 1
    print(c, '/', 10000)
    #print(len(i), len(i[0]))
    train_np_vectors4 = np.append(train_np_vectors4, [i], axis=0)

In [ ]:
train_np_vectors5 = np.array([train_matrix[0]])
print(train_np_vectors.shape)
c = 0
for i in train_matrix5:
    c += 1
    print(c, '/', 10000)
    #print(len(i), len(i[0]))
    train_np_vectors5 = np.append(train_np_vectors5, [i], axis=0)

In [ ]:
train_np_vectors6 = np.array([train_matrix[0]])
print(train_np_vectors.shape)
c = 0
for i in train_matrix6:
    c += 1
    print(c, '/', 10000)
    #print(len(i), len(i[0]))
    train_np_vectors6 = np.append(train_np_vectors6, [i], axis=0)

In [ ]:
train_np_vectors7 = np.array([train_matrix[0]])
print(train_np_vectors.shape)
c = 0
for i in train_matrix7:
    c += 1
    print(c, '/', 10000)
    #print(len(i), len(i[0]))
    train_np_vectors7 = np.append(train_np_vectors7, [i], axis=0)

In [ ]:
train_np_vectors = np.array([train_matrix[0]])

In [ ]:
train_np_vectors = np.append(train_np_vectors, train_np_vectors1[1:], axis=0)

In [ ]:
train_np_vectors = train_np_vectors[1:]

In [ ]:
train_np_vectors = np.append(train_np_vectors, train_np_vectors2[1:], axis=0)

In [ ]:
train_np_vectors = np.append(train_np_vectors, train_np_vectors3[1:], axis=0)

In [ ]:
train_np_vectors = np.append(train_np_vectors, train_np_vectors4[1:], axis=0)

In [ ]:
train_np_vectors = np.append(train_np_vectors, train_np_vectors5[1:], axis=0)

In [ ]:
train_np_vectors = np.append(train_np_vectors, train_np_vectors6[1:], axis=0)

In [ ]:
train_np_vectors = np.append(train_np_vectors, train_np_vectors7[1:], axis=0)

In [ ]:
len(train_np_vectors)

In [ ]:
len(train_matrix)

In [ ]:
np.save("aphasia_np_vectors_full5.np", train_np_vectors)

In [ ]:
np.save('aphasia_np_vectors_full5_test.np', test_np_vectors)

In [ ]:
np.save("aphasia_np_labels_full5_test.np", np.array(test_labels))
np.save("aphasia_np_labels_full5_train.np", np.array(train_labels))

In [ ]:
f = train_np_vectors.reshape(1,-1)

In [ ]:
f.shape

In [ ]:
train_np_vectors.reshape(62393, 28, 300, 1)

## Label counts of the patient-wise split (Table 2)

In [ ]:
import numpy as np

In [ ]:
train_labels = np.load("aphasia_np_labels_patientwise_train.np.npy")
test_labels = np.load("aphasia_np_labels_patientwise_test.np.npy")

In [ ]:
train_dict = dict()
for i in train_labels:
    if i in train_dict:
        train_dict[i] += 1
    else:
        train_dict[i] = 1
print(train_dict)

In [ ]:
test_dict = dict()
for i in test_labels:
    if i in test_dict:
        test_dict[i] += 1
    else:
        test_dict[i] = 1
print(test_dict)

In [ ]:
len(test_labels)